### Initial Setup

In [ ]:
import torch
import urllib
from PIL import Image
from torchvision import transforms

# Load the pre-trained ResNet18 model
model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=True)
model.eval()

# Download an example image
url, filename = ("https://github.com/pytorch/hub/raw/master/images/dog.jpg", "dog.jpg")
try: urllib.URLopener().retrieve(url, filename)
except: urllib.request.urlretrieve(url, filename)

# Preprocess the image
input_image = Image.open(filename)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = preprocess(input_image)
input_batch = input_tensor.unsqueeze(0)

# Download ImageNet labels
!wget https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt

# Read the categories
with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

Downloading: "https://github.com/pytorch/vision/zipball/v0.10.0" to /root/.cache/torch/hub/v0.10.0.zip
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 153MB/s]


--2024-04-20 22:28:26--  https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10472 (10K) [text/plain]
Saving to: ‘imagenet_classes.txt’

imagenet_classes.tx 100%[===================>]  10.23K  --.-KB/s    in 0.001s  

2024-04-20 22:28:26 (17.2 MB/s) - ‘imagenet_classes.txt’ saved [10472/10472]



### Untargeted FGSM Attack

In [ ]:
import torch.nn.functional as F

# If CUDA is available, move the data to GPU
if torch.cuda.is_available():
    input_batch = input_batch.to('cuda')
    model.to('cuda')

# Make input tensor require gradients
input_batch.requires_grad = True

# Forward pass to compute the model output
output = model(input_batch)

# Get the index of the maximum logit to represent the predicted class
initial_pred = output.argmax(dim=1)  # No need for keepdim=True here

# Ensure we are using initial_pred with the right dimensions for loss calculation
loss = F.cross_entropy(output, initial_pred)  # initial_pred does not need squeezing now

# Zero all existing gradients
model.zero_grad()

# Backward pass to calculate gradients w.r.t. input
loss.backward()
data_grad = input_batch.grad.data

# FGSM attack: Create the adversarial example by adding an epsilon-sized sign of the gradient
epsilon = 0.05
perturbed_image = input_batch + epsilon * data_grad.sign()
perturbed_image = torch.clamp(perturbed_image, 0, 1)

# Re-classify the perturbed image to see how the label has changed
output_perturbed = model(perturbed_image)
perturbed_pred = output_perturbed.argmax(dim=1, keepdim=True)  # keepdim for consistent tensor shape

# Print results
print("Original label:", categories[initial_pred.item()])
print("Perturbed label:", categories[perturbed_pred.item()])

Original label: Samoyed
Perturbed label: alp


### Least Likely FGSM

In [ ]:
import torch
import torch.nn.functional as F

# Ensure the model and input_batch are prepared and that the model is in evaluation mode
model.eval()

# Calculate the original output without any gradient computation
with torch.no_grad():
    output = model(input_batch)

# Find the least likely class for the image
least_likely_class = output.argmin(dim=1)

# Enable gradient computation for input batch for adversarial example generation
input_batch.requires_grad = True

# Compute the output again for gradient computation
output = model(input_batch)

# Compute the cross-entropy loss between the output and the least likely class
loss_ll = F.cross_entropy(output, least_likely_class)

# Zero any existing gradients before backward pass
model.zero_grad()

# Perform backward pass to compute gradients
loss_ll.backward()

# Obtain the gradients of the input batch
data_grad_ll = input_batch.grad.data

# Generate the adversarial example by subtracting epsilon times the sign of the data gradient
epsilon = 0.05
perturbed_image_ll = input_batch - epsilon * data_grad_ll.sign()
perturbed_image_ll = torch.clamp(perturbed_image_ll, 0, 1)

# Detach the perturbed image to cut it off from previous computation graphs
perturbed_image_ll = perturbed_image_ll.detach()

# Ensure no gradients are accumulated in the model
model.zero_grad()

# Classify the adversarial example and extract the new predicted class
output_perturbed_ll = model(perturbed_image_ll)
perturbed_pred_ll = output_perturbed_ll.argmax(dim=1)

# Print the results
print("Least Likely Original label:", categories[least_likely_class.item()])
print("Perturbed label from Least Likely FGSM:", categories[perturbed_pred_ll.item()])

Least Likely Original label: bittern
Perturbed label from Least Likely FGSM: vulture


### Projected Gradient Descent (PGD)

In [ ]:
import torch
import torch.nn.functional as F

# Assuming the model and input_batch have been properly initialized and set up
# Configuring PGD parameters
epsilon = 0.05
alpha = 0.01  # Small step size
num_steps = 40

# Create a copy of the original input to perturb
perturbed_image_pgd = input_batch.clone().detach()
if torch.cuda.is_available():
    perturbed_image_pgd = perturbed_image_pgd.to('cuda')

# PGD Attack loop
for _ in range(num_steps):
    perturbed_image_pgd.requires_grad = True
    output_pgd = model(perturbed_image_pgd)
    initial_pred = output_pgd.argmax(dim=1)  # Get the current predictions
    loss_pgd = F.cross_entropy(output_pgd, initial_pred)
    model.zero_grad()
    loss_pgd.backward()
    data_grad_pgd = perturbed_image_pgd.grad.data

    # Apply the perturbation using the sign of the gradient
    perturbed_image_pgd = perturbed_image_pgd + alpha * data_grad_pgd.sign()
    # Project back to the epsilon-ball and clip to valid image range
    eta = torch.clamp(perturbed_image_pgd - input_batch, min=-epsilon, max=epsilon)
    perturbed_image_pgd = torch.clamp(input_batch + eta, 0, 1).detach_()  # Detach to avoid influence from previous gradients

# Evaluate the model on the adversarially perturbed image
output_perturbed_pgd = model(perturbed_image_pgd)
perturbed_pred_pgd = output_perturbed_pgd.argmax(dim=1)

# Print results
print("Perturbed label from PGD:", categories[perturbed_pred_pgd.item()])

Perturbed label from PGD: jigsaw puzzle


### Carlini-Wagner (CW) Attack

In [ ]:
import torch.optim as optim

# Set up CW attack parameters
cw_steps = 100
cw_lr = 0.01
target_class = torch.tensor([output.argmin(dim=1).item()], device='cuda' if torch.cuda.is_available() else 'cpu')  # Targeting the least likely class

# Initialize perturbation
perturbation = torch.zeros_like(input_batch, requires_grad=True)
optimizer = optim.Adam([perturbation], lr=cw_lr)

# Perform the CW attack
for _ in range(cw_steps):
    perturbed_image_cw = input_batch + perturbation
    perturbed_image_cw = torch.clamp(perturbed_image_cw, 0, 1)  # Ensure the values stay in the image range

    output_cw = model(perturbed_image_cw)
    loss_cw = F.cross_entropy(output_cw, target_class)
    optimizer.zero_grad()
    loss_cw.backward()
    optimizer.step()

    # Optionally enforce some constraint on the perturbation
    with torch.no_grad():
        perturbation.data = torch.clamp(perturbation.data, -epsilon, epsilon)

output_perturbed_cw = model(perturbed_image_cw)
perturbed_pred_cw = output_perturbed_cw.argmax(dim=1)

# Print results
print("Target label from CW:", categories[target_class.item()])
print("Perturbed label from CW:", categories[perturbed_pred_cw.item()])

Target label from CW: bittern
Perturbed label from CW: bittern
